# Create submission files

For a simulations with ekbatch you need an init file containing the stimuli and the CVs of the regions from a CARP simulation, you will need:
-  vtx file for a stimulus
-  a set of tags with the conduction velocities

In [1]:
import json
import numpy as np
import tqdm

def json_to_init(stimuli, tag_file, json_param_file, init_file_name):

    # Read tags
    f_input = open(tag_file,"r")
    tags = json.load(f_input)
    f_input.close()

    # Read CVs
    f_input = open(json_param_file,"r")
    params = json.load(f_input)
    f_input.close()

    tags_ventricles_names = ["LV", "RV"]
    CV_ventricle_name = "CV_ventricles"
    if not CV_ventricle_name in params["EP"].keys():
        CV_ventricle_name = "CV_f_v"
    k_ventricles_name = "ani_ratio_v"
    if not k_ventricles_name in params["EP"].keys():
        k_ventricles_name = "ani_ratio_ventricles"

    tags_FEC_names = ["FEC_LV", "FEC_RV", "FEC_SV"]
    k_FEC_name = "k_FEC"

    tags_atria_names = ["LA", "RA"]
    CV_atria_name = "CV_atria"
    if not CV_atria_name in params["EP"].keys():
        CV_atria_name = "CV_f_a"
    k_atria_name = "ani_ratio_v"
    if not k_atria_name in params["EP"].keys():
        k_atria_name = "ani_ratio_atria"

    tags_bachmann_names = ["BB"]
    k_BB_name = "k_BB"

    vtx = []
    nVtx = 0

    for vtxFile in stimuli:
        temp = np.loadtxt(vtxFile, dtype=int, skiprows=2, ndmin=1)
        vtx.append(temp)
        nVtx += temp.shape[0]

    # write .init file
    f = open(init_file_name,'w')

    # header
    f.write('vf:0 vs:0 vn:0 vPS:0\n') # Default properties for tags not specified
    f.write('retro_delay:0 antero_delay:0\n') # If there's no 1D purkinje system, it's ignored.
    # number of stimuli and regions
    f.write('%d %d\n' % (int(nVtx), int(len(tags_ventricles_names)) + len(tags_FEC_names) + len(tags_atria_names) + len(tags_bachmann_names)))
    # stimulus
    for i in range(len(vtx)):
        if len(vtx[i]) == 1:
            f.write('%d %f\n' % (vtx[i],0))
        else:
            for n in vtx[i]:
                f.write('%d %f\n' % (int(n),0))
                
    return_tags_str = ''
    # ek regions
    for i,tag_name in enumerate(tags_ventricles_names):
        f.write('%d %f %f %f\n' % (int(tags[tag_name]), 
                                   float(params["EP"][CV_ventricle_name]), 
                                   float(params["EP"][CV_ventricle_name])*float(params["EP"][k_ventricles_name]), 
                                   float(params["EP"][CV_ventricle_name])*float(params["EP"][k_ventricles_name])))
        return_tags_str += ',' + str(int(tags[tag_name]))

    for i,tag_name in enumerate(tags_FEC_names):
        f.write('%d %f %f %f\n' % (int(tags[tag_name]), 
                                   float(params["EP"][CV_ventricle_name])*float(params["EP"][k_FEC_name]), 
                                   float(params["EP"][CV_ventricle_name])*float(params["EP"][k_FEC_name]), 
                                   float(params["EP"][CV_ventricle_name])*float(params["EP"][k_FEC_name])))
        return_tags_str += ',' + str(int(tags[tag_name]))

    for i,tag_name in enumerate(tags_atria_names):
        f.write('%d %f %f %f\n' % (int(tags[tag_name]), 
                                   float(params["EP"][CV_atria_name]), 
                                   float(params["EP"][CV_atria_name])*float(params["EP"][k_atria_name]), 
                                   float(params["EP"][CV_atria_name])*float(params["EP"][k_atria_name])))
        return_tags_str += ',' + str(int(tags[tag_name]))

    for i,tag_name in enumerate(tags_bachmann_names):
        f.write('%d %f %f %f\n' % (int(tags[tag_name]), 
                                   float(params["EP"][CV_atria_name])*float(params["EP"][k_BB_name]), 
                                   float(params["EP"][CV_atria_name])*float(params["EP"][k_BB_name]), 
                                   float(params["EP"][CV_atria_name])*float(params["EP"][k_BB_name])))
        return_tags_str += ',' + str(int(tags[tag_name]))

    f.close()
    
    return return_tags_str[1:]

In [2]:
import os

heart_folder = "/data/HCM/1/"
mesh_folder = "/media/croderog/SeagateExpansionDrive/HCM/1"
scenario = f"51"
Nsim = 120

stimuli = [f'{mesh_folder}/sims_folder/fascicles_lv.vtx',
                f'{mesh_folder}/sims_folder/fascicles_rv.vtx',
                f'{mesh_folder}/sims_folder/SAN.vtx']

json_param_path        = f'{heart_folder}/scenarios/{scenario}/json_files/'
tag_file        = f'{json_param_path}/tags_EP.json'
init_file_path  = f'{heart_folder}/scenarios/{scenario}/data/init_files'

os.system("mkdir -p " + init_file_path)

for sim_num in range(Nsim):
    tags_activated = json_to_init(stimuli=stimuli,
                tag_file=tag_file,
                json_param_file=os.path.join(json_param_path,str(sim_num) + '.json'),
                init_file_name=os.path.join(init_file_path,str(sim_num) + '.init')
                )

# Run simulations

In [6]:


sims_folder = f'{heart_folder}/scenarios/{scenario}/simulations'

meshname = f'{mesh_folder}/sims_folder/myocardium_AV_FEC_BB_lvrv'


cmd = ['ekbatch',meshname]
init_cmd = ','.join([os.path.join(init_file_path,str(sim_num)) for sim_num in range(Nsim)])

os.system(' '.join(cmd+[init_cmd] + [tags_activated]))

os.makedirs(sims_folder,exist_ok=True)
for sim_num in range(Nsim):
    os.system('mv ' + os.path.join(init_file_path,str(sim_num) + '.dat ') + sims_folder)


Executable ID: ICL_LHR_CARPENTRY
Found license file path: /home/croderog/software/CARPentry_ICL_latest/license/license.bin
Using OpenMP parallelization with 24 threads.
Reading mesh ..
Reading elements (bin):                           [==============================]
Reading points (bin):                             [==============================]
Reading fibers (bin):                             [=====================         ]

mv: cannot stat '/data/HCM/1//scenarios/51/data/init_files/0.dat': No such file or directory
mv: cannot stat '/data/HCM/1//scenarios/51/data/init_files/1.dat': No such file or directory
mv: cannot stat '/data/HCM/1//scenarios/51/data/init_files/2.dat': No such file or directory
mv: cannot stat '/data/HCM/1//scenarios/51/data/init_files/3.dat': No such file or directory
mv: cannot stat '/data/HCM/1//scenarios/51/data/init_files/4.dat': No such file or directory
mv: cannot stat '/data/HCM/1//scenarios/51/data/init_files/5.dat': No such file or directory
mv: cannot stat '/data/HCM/1//scenarios/51/data/init_files/6.dat': No such file or directory
mv: cannot stat '/data/HCM/1//scenarios/51/data/init_files/7.dat': No such file or directory
mv: cannot stat '/data/HCM/1//scenarios/51/data/init_files/8.dat': No such file or directory
mv: cannot stat '/data/HCM/1//scenarios/51/data/init_files/9.dat': No such file or directory
mv: cannot stat '/data/HCM/1//scenarios/51/data/init_files/10.dat': No

# Extract the output

In [7]:
# Extracted from Marina's library

def electrophysiology_output(basefolder,
							 elem_file,
							 tags,
	   						 start_sample=0,
	   						 last_sample=1,
	   						 output_file='Y.txt'):

	print('Reading mesh elem file...')
	elem = np.loadtxt(elem_file,dtype=int,usecols=[1,2,3,4,5],skiprows=1)
	print('Done.')

	V_EIDX = np.where(np.isin(elem[:,-1],tags["ventricles"]+tags["fast_endo"])==1)[0]
	A_EIDX = np.where(np.isin(elem[:,-1],tags["atria"]+tags["bachmann_bundle"])==1)[0]

	V_VTX = np.unique(elem[V_EIDX,0:4].flatten())
	A_VTX = np.unique(elem[A_EIDX,0:4].flatten())

	output = np.zeros((last_sample-start_sample+1,2))

	count = 0
	t = tqdm.trange(len(range(start_sample,last_sample+1)), desc='Bar desc', leave=True,colour='#FFFF00')
	for i in t:
		t.set_description('Computing output for '+str(i)+'.dat...')
		AT=np.loadtxt(os.path.join(basefolder,str(i)+".dat"),dtype=float)
		if (np.min(AT[V_VTX]<0)):
			raise Exception("The ventricles contain a negative activation time.")
		if (np.min(AT[A_VTX]<0)):
			raise Exception("The atria contain a negative activation time.")
			
		output[count,0] = np.max(AT[A_VTX])-np.min(AT[A_VTX])
        
		output[count,1] = np.max(AT[V_VTX])-np.min(AT[V_VTX])
		count += 1

	np.savetxt(output_file,output,fmt="%g")

In [8]:
import json
import numpy as np
import os

basefolder = sims_folder
elem_file = f"{meshname}.elem"

f_input = open(tag_file,"r")
tags = json.load(f_input)
f_input.close()


tags_modified = tags.copy()
tags_modified["ventricles"] = [tags_modified["LV"], tags_modified["RV"]]
tags_modified["fast_endo"] = [tags_modified["FEC_RV"], tags_modified["FEC_SV"]]
tags_modified["atria"] = [tags_modified["LA"], tags_modified["RA"]]
tags_modified["bachmann_bundle"] = [tags_modified["BB"]]

output_path = f'{heart_folder}/scenarios/{scenario}/data'

electrophysiology_output(basefolder=basefolder,
							elem_file=elem_file,
							tags=tags_modified,
							start_sample=0,
							last_sample=Nsim-1,
							output_file=os.path.join(output_path,'Y.txt'))

Reading mesh elem file...
Done.


Computing output for 119.dat...: 100%|██████████| 120/120 [05:11<00:00,  2.60s/it]


# Make animation of the EP simulation

In [1]:
import json
import math
import numpy as np
import pyvista as pv
import tqdm
import vtk

import pyvista as pv
import numpy as np
import multiprocessing
from functools import partial
import os


def _screenshot_worker(frame_data,
                       mesh,
                       output_dir,
                       camera_settings,
                       inactive_color,
                       active_color,
                       opacity,
                       fig_w,
                       fig_h,
                       view):

    t, binary_vector = frame_data
    mesh_copy = mesh.copy(deep=True)
    mesh_copy.point_data["at"] = binary_vector

    plotter = pv.Plotter(off_screen=True)
    plotter.background_color = 'white'

    plotter.add_mesh(mesh_copy,
                     scalars="at",
                     opacity=opacity,
                     cmap=[inactive_color, active_color],
                     clim=[0., 1.],
                     show_scalar_bar=False)

    # Set camera
    cam = plotter.camera
    cam.azimuth = camera_settings[view]["azimuth"]
    cam.elevation = camera_settings[view]["elevation"]
    cam.roll = camera_settings[view]["roll"]

    plotter.add_title(f"time = {t} ms", font_size=12,
                      font="arial", color="black")

    screenshot_path = os.path.join(output_dir, f"act_{t:03d}.png")
    plotter.screenshot(screenshot_path, window_size=[fig_w, fig_h])
    plotter.close()



def render_activation_video_parallel(mesh,
                                     act_vector,
                                     output_dir,
                                     camera_settings,
                                     inactive_color="lightgray",
                                     active_color="firebrick",
                                     fig_w=1200,
                                     fig_h=1200,
                                     opacity=1.0,
                                     view="anterior",
                                     num_workers=None):

    os.makedirs(output_dir, exist_ok=True)

    # Compute activation time range
    act_vector = act_vector.copy()
    t0 = 0
    tend = int(np.ceil(np.max(act_vector[act_vector < 1e6])))
    act_vector[act_vector < 0] = tend + 10

    # Prepare data for each frame
    frame_data_list = [(t, (act_vector <= t).astype(np.uint8)) for t in range(t0, tend + 1)]

    # Use all CPUs if not specified
    if num_workers is None:
        num_workers = multiprocessing.cpu_count()

    with multiprocessing.Pool(num_workers) as pool:
        worker_fn = partial(
            _screenshot_worker,
            mesh=mesh,
            output_dir=output_dir,
            camera_settings=camera_settings,
            inactive_color=inactive_color,
            active_color=active_color,
            opacity=opacity,
            fig_w=fig_w,
            fig_h=fig_h,
            view=view
        )
        list(pool.imap(worker_fn, frame_data_list))


def read_elem(filename,el_type='Tt',tags=True):
	print('Reading '+filename+'...')

	if el_type=='Tt':
		if tags:
			return np.loadtxt(filename, dtype=int, skiprows=1, usecols=(1,2,3,4,5))
		else:
			filtered_lines = []
			with open(filename, 'r') as infile:
				first_line = True
				for line in infile:
					if first_line:
						first_line = False
						continue
					else:
					# Split the line into columns
						columns = line.split()
						# Check if the number of columns is 6
						if len(columns) == 6:
							filtered_lines.append(columns[1:5])
						else:
							break
    
			# Convert the filtered lines to a numpy array
			# Skipping the first row (header) and using specific columns
			data = np.array(filtered_lines, dtype=int)
			return data
			# return np.loadtxt(filename, dtype=int, skiprows=1, usecols=(1,2,3,4))
	elif el_type=='Tr':
		if tags:
			return np.loadtxt(filename, dtype=int, skiprows=1, usecols=(1,2,3,4))
		else:
			return np.loadtxt(filename, dtype=int, skiprows=1, usecols=(1,2,3))
	elif el_type=='Ln':
		if tags:
			return np.loadtxt(filename, dtype=int, skiprows=1, usecols=(1,2,3))
		else:
			return np.loadtxt(filename, dtype=int, skiprows=1, usecols=(1,2))
	else:
		raise Exception('element type not recognised. Accepted: Tt, Tr, Ln')

def carp_to_pyvista(meshname):

	pts = np.loadtxt(meshname+'.pts', dtype=float, skiprows=1)
	elem = read_elem(meshname+'.elem',el_type='Tt',tags=False)

	tets = np.column_stack((np.ones((elem.shape[0],),dtype=int)*4,elem)).flatten()
	cell_type = np.ones((elem.shape[0],),dtype=int)*vtk.VTK_TETRA	

	plt_msh = pv.UnstructuredGrid(tets,cell_type,pts)

	return plt_msh

def numpy_hook(dct):
	for key, value in dct.items():
		if isinstance(value, list):
			value = np.array(value)
			dct[key] = value
	return dct

def load_json(filename):
	print('Reading '+filename+'...')

	dct = {}
	with open(filename, "r") as f:
		dct = json.load(f, object_hook=numpy_hook)
	return dct

def print_screenshot_video(plt_msh,
						   binary_vector,
						   screenshot_name,
						   camera_settings,
						   title=None,
						   fig_w=1200,
						   fig_h=1200,
						   inactive_color="gray",
						   active_color="darkred",
						   view="anterior",
						   opacity=1.0):

	plotter = pv.Plotter(off_screen=True)
	plotter.background_color = 'white'

	plt_msh.point_data["at"] = binary_vector

	msh = plotter.add_mesh(plt_msh,opacity=opacity,
						   scalars="at",
						   cmap=[inactive_color,active_color],
						   clim=np.array([0.,1.]))

	plotter.remove_scalar_bar()

	plotter.camera.azimuth = camera_settings[view]["azimuth"]
	plotter.camera.elevation = camera_settings[view]["elevation"]
	plotter.camera.roll = camera_settings[view]["roll"]

	plotter.add_title(title,
					  font_size=12,
					  font="arial",
					  color="black")
	print("Printing...")
	plotter.screenshot(filename=screenshot_name, 
					   transparent_background=None, 
					   return_img=True,
					   window_size=[fig_w,fig_h])
	print("Printed")
	plotter.close()

def make_activation_video(meshname,
						  activation_file,
						  video_folder,
						  camera_file,
					 	  inactive_color="lightgray",
					 	  active_color="firebrick",
					 	  view="anterior",
						  opacity=1.0):
	
	camera_settings = load_json(camera_file)

	plt_msh = carp_to_pyvista(meshname)
	
	act = np.loadtxt(activation_file,dtype=float)
	
	t0 = 0 
	tend = math.ceil(np.max(act[act < 1e6]))

	act[act < 0] = tend+10


	count = 0
	print(t0)
	print(tend)
	for t in tqdm.tqdm(range(t0,tend+1)):

		binary_vector = (act<=t)

		print_screenshot_video(plt_msh,
					           binary_vector,
					           video_folder+"/act_{:03d}.png".format(count),
					           camera_settings,
					           title="time = "+str(t)+" ms",
					           fig_w=1200,
					           fig_h=1200,
					           inactive_color=inactive_color,
					           active_color=active_color,
					           view=view,
							   opacity=opacity)

		count += 1

In [2]:
case="5"
folder="40"



make_activation_video(meshname = f"/media//croderog/SeagateExpansionDrive/HCM/{case}/sims_folder/myocardium_AV_FEC_BB_lvrv",
						activation_file = f"/media/croderog/SeagateExpansionDrive/HCM/5/scenarios/40/simulations/0.dat",
						video_folder=f"/media/croderog/Bob/HCM/figures",
						camera_file="/media/croderog/Bob/HCM/figures/camera_settings.json",
						inactive_color="whitesmoke",
						active_color="goldenrod" , # dark yellow
						view="anterior",
						opacity=0.8)

Reading /media/croderog/Bob/HCM/figures/camera_settings.json...
Reading /media//croderog/SeagateExpansionDrive/HCM/5/sims_folder/myocardium_AV_FEC_BB_lvrv.elem...
0
327


  0%|          | 0/328 [00:00<?, ?it/s]

Printing...


  0%|          | 1/328 [00:07<43:03,  7.90s/it]

Printed
Printing...


...

Printed
Printing...


100%|█████████▉| 327/328 [40:54<00:07,  7.08s/it]

Printed
Printing...


100%|██████████| 328/328 [41:01<00:00,  7.51s/it]

Printed
